In [14]:
import os
import json
import re
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from utils import l2_normalize, cosine_similarity

MODEL_PATH = "afs_data/model/Qwen3-Embedding-8B"
EMB_MODEL = SentenceTransformer(MODEL_PATH)


def encode_fn(texts: List[str], batch_size=64, normalize_embeddings=False, show_progress_bar=False) -> np.ndarray:
    """向量模型编码函数。"""
    embeddings = EMB_MODEL.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=show_progress_bar,
    )
    return np.asarray(embeddings)

Loading weights: 100%|██████████| 398/398 [00:10<00:00, 37.19it/s]


In [16]:
def encode_fn(texts: List[str], batch_size=64, normalize_embeddings=False, show_progress_bar=False) -> np.ndarray:
    """向量模型编码函数。"""
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=normalize_embeddings,
        show_progress_bar=show_progress_bar,
    )
    return np.asarray(embeddings)

In [17]:
res = encode_fn(["aab", "bbc"], batch_size=2, normalize_embeddings=True)

In [18]:
def l2_normalize(vectors: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2 归一化，归一化后点积等价于 cosine similarity。"""
    if vectors.ndim == 1:
        vectors = vectors.reshape(1, -1)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(norms, eps)
l2_normalize(res)

NORMALIZE_PUNCT_RE = re.compile(r"[，。！？、,.!?;；:：\"'“”‘’（）()\[\]{}\s]+")
TOKEN_RE = re.compile(r"[\u4e00-\u9fff]|[A-Za-z0-9_]+")

def normalize_text(text: str) -> str:
    return NORMALIZE_PUNCT_RE.sub("", text.lower())

In [26]:

def cosine_similarity(v1: np.ndarray, v2: np.ndarray):
    dot_product = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

class DescriptionComparator:
    """Compares original and optimized descriptions with exact, normalized and semantic checks."""

    def __init__(self, encoder: Any | None = None, semantic_threshold: float = 0.985) -> None:
        self.encode_fn = encode_fn or encoder
        self.semantic_threshold = semantic_threshold

    def compare(self, original_description: str, optimized_description: str) -> dict[str, Any]:
        same_exact = original_description == optimized_description
        same_normalized = normalize_text(original_description) == normalize_text(optimized_description)
        embeddings = self.encode_fn([original_description, optimized_description], batch_size=2)
        semantic_similarity = 1.0 if same_normalized else cosine_similarity(embeddings[0], embeddings[1])

        changed = not (same_exact or same_normalized or semantic_similarity > self.semantic_threshold)
        
        return {
            "same_exact": same_exact,
            "same_normalized": same_normalized,
            "semantic_similarity": round(semantic_similarity, 6),
            "changed": changed,
        }

dc = DescriptionComparator(encode_fn)
dc.compare("adfb", "sdf")

{'same_exact': False,
 'same_normalized': False,
 'semantic_similarity': np.float32(0.747937),
 'changed': True}

In [22]:
original_description = "adfb"
optimized_description = "sdf"
embeddings = encode_fn([original_description, optimized_description], batch_size=2)
embeddings

array([[ 0.03808594,  0.00927734,  0.00366211, ..., -0.00183105,
        -0.00482178,  0.00482178],
       [ 0.02758789, -0.00131226, -0.00059128, ..., -0.00741577,
        -0.00628662,  0.0133667 ]], shape=(2, 4096), dtype=float32)

In [25]:
cosine_similarity(embeddings[0], embeddings[1])

np.float32(0.74793684)